# pyRTE runs for WUS Warming paper
Based off of pyRTE quick start notebook

## Overview

PyRTE-RRTMGP provides a flexible and efficient framework for computing radiative fluxes in planetary atmospheres. 

See the [documentation](https://pyrte-rrtmgp.readthedocs.io/en/latest/) for more information.

# Initialization




In [215]:
do_plots = True
from matplotlib import pyplot as plt
import numpy as np

def fix_coords_lon(ds):
    ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180)).sortby(['longitude','latitude'])
    return ds
wusbox = [-125, -102, 32,  49]

import json
import pandas as pd

In [216]:
def detrend_dim(da, dim, deg=1):
    # detrend along a single dimension
    p = da.polyfit(dim=dim, deg=deg)
    fit = xr.polyval(da[dim], p.polyfit_coefficients)
    return da - fit

In [217]:
import scipy

## Import dependencies

In [218]:
%matplotlib inline

from dask.diagnostics import ProgressBar
import xarray as xr

if do_plots: import matplotlib.pyplot as plt

## Import pyRTE entitites 

(The organization is a work in progress) 

In [219]:
from pyrte_rrtmgp.rrtmgp_data_files import (
    CloudOpticsFiles,
    GasOpticsFiles,
)
from pyrte_rrtmgp.examples import (
    compute_RCE_clouds,
    compute_RCE_profiles,
    ALLSKY_EXAMPLES,
    load_example_file,
)
from pyrte_rrtmgp import rte
from pyrte_rrtmgp.rrtmgp import GasOptics, CloudOptics

## Initialize gas and cloud optics 

In [220]:
cloud_optics_lw = CloudOptics(
    cloud_optics_file=CloudOpticsFiles.LW_BND
)
gas_optics_lw = GasOptics(
    gas_optics_file=GasOpticsFiles.LW_G256
)

cloud_optics_sw = CloudOptics(
    cloud_optics_file=CloudOpticsFiles.SW_BND
)
gas_optics_sw = GasOptics(
    gas_optics_file=GasOpticsFiles.SW_G224
)

The optics classes are `xarray Datasets` but the underlying data isn't meant to be accessed directly.

In [221]:
cloud_optics_lw, gas_optics_lw

(<pyrte_rrtmgp.rrtmgp.CloudOptics at 0x7f52fea3d040>,
 <pyrte_rrtmgp.rrtmgp.LWGasOptics at 0x7f52fea51100>)

## Temperature, humidity, composition

The routine `compute_RCE_profiles()` packaged with `pyRTE_RRTMGP` computes temperature, pressure, and humidity profiles following a moist adibat. The concentrations of other gases are also needed.

## Open ERA5 data

In [15]:
# gz and temperature profiles
gzt_wus = xr.open_dataset('/d5/tessj/data/ERA5/pressure_level/monthly/gzt_monthly_1980_2024_WUS.nc')
gzt_wus = gzt_wus.weighted(weights=np.cos(np.deg2rad(gzt_wus.latitude))).mean(['latitude', 'longitude']).rename({'valid_time':'time'})

In [16]:
# ozone and humidity profiles
o3q_wus = xr.open_dataset('/d5/tessj/data/ERA5/pressure_level/monthly/o3q_monthly_1980_2024_WUS.nc')
o3q_wus = o3q_wus.weighted(weights=np.cos(np.deg2rad(o3q_wus.latitude))).mean(['latitude', 'longitude']).rename({'valid_time':'time'})

In [17]:
# skin temp
skt_wus = xr.open_dataset('/d5/tessj/data/ERA5/pressure_level/monthly/skt_monthly_1980_2024_WUS.nc')
skt_wus = skt_wus.weighted(weights=np.cos(np.deg2rad(skt_wus.latitude))).mean(['latitude', 'longitude']).rename({'valid_time':'time'})
skt_wus

<xarray.Dataset> Size: 17kB
Dimensions:  (time: 540)
Coordinates:
  * time     (time) datetime64[ns] 4kB 1980-01-01 1980-02-01 ... 2024-12-01
    expver   (time) <U4 9kB '0001' '0001' '0001' '0001' ... '0001' '0001' '0001'
    number   int64 8B 0
Data variables:
    skt      (time) float64 4kB 272.4 275.4 277.3 283.6 ... 288.1 277.9 276.2
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-12T22:39 GRIB to CDM+CF via cfgrib-0.9.1...

In [10]:
era5_lwd_clr = (1/86400)*xr.open_dataset('/d4/tessj/data/ERA5/single_level/monthly/ds_slwd_clr_1940_2025.nc')
era5_lwd_clr = fix_coords_lon(era5_lwd_clr).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()
era5_lwd_clr_jfm = era5_lwd_clr.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')

/tmp/ipykernel_27844/949569015.py:2: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5_lwd_clr = fix_coords_lon(era5_lwd_clr).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()


In [125]:
era5_lwd_clr = xr.open_dataset('/d5/tessj/data/ERA5/single_level/monthly/ds_msdwlwrf_1940_2025.nc')
era5_lwd_clr = fix_coords_lon(era5_lwd_clr).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()
era5_lwd_clr_jfm = era5_lwd_clr.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')

/tmp/ipykernel_27844/1325322834.py:2: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5_lwd_clr = fix_coords_lon(era5_lwd_clr).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()


In [24]:
era5_clt = xr.open_dataset('/d4/tessj/data/ERA5/single_level/monthly/ds_tcc_1940_2025.nc')
era5_clt = fix_coords_lon(era5_clt).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()
era5_clt_jfm = era5_clt.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')

/tmp/ipykernel_22931/590457133.py:2: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5_clt = fix_coords_lon(era5_clt).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()


In [11]:
era5_lwd = xr.open_dataset('/d4/tessj/data/ERA5/single_level/monthly/ds_msdwlwrf_1940_2025.nc')
era5_lwd = fix_coords_lon(era5_lwd).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()
era5_lwd_jfm = era5_lwd.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')

/tmp/ipykernel_27844/1058644502.py:2: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5_lwd = fix_coords_lon(era5_lwd).rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).sel(lon=slice(wusbox[0],wusbox[1]), lat=slice(wusbox[2], wusbox[3])).drop(['number', 'expver']).load()


In [12]:
landmask_era5 = fix_coords_lon(xr.open_dataarray('/home/tessj/wildfire-clim/land_sea_mask.nc').isel(time=-1).load()).rename({'latitude':'lat', 'longitude':'lon'}).drop(['expver','time'])
landmask_era5 = landmask_era5.where(landmask_era5>0.4, np.nan)
landmask_era5 = landmask_era5/landmask_era5

/tmp/ipykernel_27844/475667704.py:1: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  landmask_era5 = fix_coords_lon(xr.open_dataarray('/home/tessj/wildfire-clim/land_sea_mask.nc').isel(time=-1).load()).rename({'latitude':'lat', 'longitude':'lon'}).drop(['expver','time'])


In [13]:
era5_weights = np.cos(np.deg2rad(era5_lwd_jfm.lat))
era5_wusavg_lwd = era5_lwd_jfm.where(landmask_era5==landmask_era5).weighted(era5_weights).mean(['lat','lon']).sel(year=slice(1980,2024)).avg_sdlwrf.values

In [160]:
era5_lwd_clr_51_80_m_clim = era5_lwd_clr.sel(time=slice('1951-01-01', '1980-12-31')).groupby('time.month').mean('time')
era5_lwd_clr_51_80_m_anom = era5_lwd_clr.groupby('time.month') - era5_lwd_clr_51_80_m_clim

In [166]:
era5_lwd_clr_51_80_m_anom_jfm = era5_lwd_clr_51_80_m_anom.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')
era5_wusavg_lwd_clr_51_80_m_anom = era5_lwd_clr_51_80_m_anom_jfm.where(landmask_era5==landmask_era5).weighted(era5_weights).mean(['lat','lon']).sel(year=slice(1980,2024)).avg_sdlwrfcs

In [128]:
era5_wusavg_lwd_clr = era5_lwd_clr_jfm.where(landmask_era5==landmask_era5).weighted(era5_weights).mean(['lat','lon']).sel(year=slice(1980,2024)).avg_sdlwrfcs.values

In [18]:
era5_jfm = xr.merge([
    gzt_wus.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time'),
    o3q_wus.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time'),  
])

/tmp/ipykernel_27844/3383619829.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  era5_jfm = xr.merge([


In [19]:
skt_wus_jfm = skt_wus.resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')

### ERA5 runs

In [176]:
# Create atmospheric profiles and gas concentrations

def make_profiles(ncol=24, nlay=72):
    atmosphere = compute_RCE_profiles(300, ncol, nlay)

    # Add other gas values
    gas_values = {
        "co2": 348e-6,
        "ch4": 1650e-9,
        "n2o": 306e-9,
        "n2": 0.7808,
        "o2": 0.2095,
        "co": 0.0,
    }

    for gas_name, value in gas_values.items():
        atmosphere[gas_name] = value

    return atmosphere


atmosphere = make_profiles()

In [177]:
era5_jfm['pres_layer']=era5_jfm.pressure_level*100
era5_atmosphere = era5_jfm.assign_coords({'pressure_level':atmosphere.layer.values[:21]}).rename({'pressure_level':'layer'})
era5_atmosphere['h2o'] = (era5_atmosphere.q/((1 - era5_atmosphere.q)*0.622))
era5_atmosphere['surface_temperature'] = skt_wus_jfm.skt


In [70]:
era5_h2o_trend = ((era5_atmosphere_layers.polyfit(dim='year', deg=1).sel(degree=1).h2o_polyfit_coefficients * era5_atmosphere_layers.year))

In [178]:
era5_atmosphere_layer = era5_atmosphere.isel(layer=slice(1, None, 2)).assign_coords({'layer':np.arange(0,10)}).rename({'t':'temp_layer'})
era5_atmosphere_level = era5_atmosphere.isel(layer=slice(0, None, 2)).assign_coords({'layer':np.arange(0,11)}).rename({'layer':'level', 't':'temp_level', 'pres_layer':'pres_level'})
era5_atmosphere_level = era5_atmosphere_level.drop(['z','o3', 'surface_temperature', 'q','h2o'])

era5_atmosphere_layers = xr.merge([era5_atmosphere_layer, era5_atmosphere_level])
era5_atmosphere_layers = era5_atmosphere_layers.drop_vars(['z', 'q']).drop(['number'])

era5_atmosphere_layers = xr.concat([era5_atmosphere_layers.assign_coords({'column':0}),era5_atmosphere_layers.assign_coords({'column':1})], dim='column')
era5_atmosphere_layers['o3'] = era5_atmosphere_layers.o3 * 0.60345
del era5_atmosphere_layers.o3.attrs['units']
del era5_atmosphere_layers.h2o.attrs['units']

#era5_atmosphere_layers_v1 = xr.zeros_like(atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))) + era5_atmosphere_layers

/tmp/ipykernel_27844/97906269.py:3: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5_atmosphere_level = era5_atmosphere_level.drop(['z','o3', 'surface_temperature', 'q','h2o'])
/tmp/ipykernel_27844/97906269.py:6: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5_atmosphere_layers = era5_atmosphere_layers.drop_vars(['z', 'q']).drop(['number'])


In [102]:
ex_atmosphere = make_profiles()

In [24]:
# lwd with 1980 temperature prescribed
lwd_sfc_clr_yr_temp1980 = []
for year in era5_atmosphere_layers.year:
    print(year.item())
    era5_atmosphere_layers_year = era5_atmosphere_layers.sel(year=year)
    era5_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
    era5_atmosphere_layers_v2['pres_level'] = xr.ones_like(era5_atmosphere_layers_v2['pres_level']) * era5_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
    era5_atmosphere_layers_v2['pres_layer'] = era5_atmosphere_layers_year['pres_layer']
    
    if year == 1980:
        era5_atmosphere_layers_v2['temp_level'] = xr.ones_like(era5_atmosphere_layers_v2['temp_level']) * era5_atmosphere_layers_year['temp_level'].values
        era5_atmosphere_layers_v2['temp_layer'] = era5_atmosphere_layers_year['temp_layer']
        era5_atmosphere_layers_v2['surface_temperature'] = era5_atmosphere_layers_year['surface_temperature']

        era5_atmosphere_layers_v2_templevel_1980 = era5_atmosphere_layers_v2['temp_level']
        era5_atmosphere_layers_v2_templayer_1980 = era5_atmosphere_layers_v2['temp_layer']
        era5_atmosphere_layers_v2_surftemp_1980 = era5_atmosphere_layers_v2['surface_temperature']
    else:
        era5_atmosphere_layers_v2['temp_level'] = era5_atmosphere_layers_v2_templevel_1980
        era5_atmosphere_layers_v2['temp_layer'] = era5_atmosphere_layers_v2_templayer_1980     
        era5_atmosphere_layers_v2['surface_temperature'] = era5_atmosphere_layers_v2_surftemp_1980     
                
    #era5_atmosphere_layers_v2['temp_level'] = xr.ones_like(era5_atmosphere_layers_v2['temp_level']) * era5_atmosphere_layers_year['temp_level'].values
    era5_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_year['o3']
    #era5_atmosphere_layers_v2['temp_layer'] = era5_atmosphere_layers_year['temp_layer']
    #era5_atmosphere_layers_v2['surface_temperature'] = era5_atmosphere_layers_year['surface_temperature']
    era5_atmosphere_layers_v2['h2o'] = era5_atmosphere_layers_year['h2o']

    optical_props = gas_optics_lw.compute(
        era5_atmosphere_layers_v2, 
        add_to_input=False,
    )

    optical_props["surface_emissivity"] = 0.96

    clr_fluxes = optical_props.rte.solve(add_to_input=False)
    lwd_sfc_clr_yr_temp1980.append(clr_fluxes.sel(level=0, column=0).lw_flux_down.values.item())

1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [106]:
# detrend temperatures
lwd_sfc_clr_yr_tempdetrend = []
    
era5_templevel_mean = era5_atmosphere_layers['temp_level'].mean('year')
era5_templevel_detrend = detrend_dim(era5_atmosphere_layers['temp_level'], dim='year') + era5_templevel_mean
era5_atmosphere_layers['temp_level'] = era5_templevel_detrend

era5_templayer_mean = era5_atmosphere_layers['temp_layer'].mean('year')
era5_templayer_detrend = detrend_dim(era5_atmosphere_layers['temp_layer'], dim='year') + era5_templayer_mean
era5_atmosphere_layers['temp_layer'] = era5_templayer_detrend

era5_surftemp_mean = era5_atmosphere_layers['surface_temperature'].mean('year')
era5_surftemp_detrend = detrend_dim(era5_atmosphere_layers['surface_temperature'], dim='year') + era5_surftemp_mean
era5_atmosphere_layers['surface_temperature'] = era5_surftemp_detrend   

for year in era5_atmosphere_layers.year:
    print(year.item())
 
    
    era5_atmosphere_layers_year = era5_atmosphere_layers.sel(year=year)

    
    era5_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
    era5_atmosphere_layers_v2['pres_level'] = xr.ones_like(era5_atmosphere_layers_v2['pres_level']) * era5_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
    era5_atmosphere_layers_v2['pres_layer'] = era5_atmosphere_layers_year['pres_layer']
                
    era5_atmosphere_layers_v2['temp_level'] = xr.ones_like(era5_atmosphere_layers_v2['temp_level']) * era5_atmosphere_layers_year['temp_level'].values
    era5_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_year['o3']
    era5_atmosphere_layers_v2['temp_layer'] = era5_atmosphere_layers_year['temp_layer']
    era5_atmosphere_layers_v2['surface_temperature'] = era5_atmosphere_layers_year['surface_temperature']
    era5_atmosphere_layers_v2['h2o'] = era5_atmosphere_layers_year['h2o']

    optical_props = gas_optics_lw.compute(
        era5_atmosphere_layers_v2, 
        add_to_input=False,
    )

    optical_props["surface_emissivity"] = 0.96

    clr_fluxes = optical_props.rte.solve(add_to_input=False)
    lwd_sfc_clr_yr_tempdetrend.append(clr_fluxes.sel(level=0, column=0).lw_flux_down.values.item())

1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [25]:
# keep 1980 q prescribed
lwd_sfc_clr_yr_q1980 = []
for year in era5_atmosphere_layers.year:
    print(year.item())
    era5_atmosphere_layers_year = era5_atmosphere_layers.sel(year=year)
    era5_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
    era5_atmosphere_layers_v2['pres_level'] = xr.ones_like(era5_atmosphere_layers_v2['pres_level']) * era5_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
    era5_atmosphere_layers_v2['pres_layer'] = era5_atmosphere_layers_year['pres_layer']
    
    if year == 1980:
        era5_atmosphere_layers_v2['h2o'] = xr.ones_like(era5_atmosphere_layers_v2['h2o']) * era5_atmosphere_layers_year['h2o'].values
        era5_atmosphere_layers_v2_h2o_1980 = era5_atmosphere_layers_v2['h2o']
    else:
        era5_atmosphere_layers_v2['h2o'] = era5_atmosphere_layers_v2_h2o_1980  
                
    era5_atmosphere_layers_v2['temp_level'] = xr.ones_like(era5_atmosphere_layers_v2['temp_level']) * era5_atmosphere_layers_year['temp_level'].values
    era5_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_year['o3']
    era5_atmosphere_layers_v2['temp_layer'] = era5_atmosphere_layers_year['temp_layer']
    era5_atmosphere_layers_v2['surface_temperature'] = era5_atmosphere_layers_year['surface_temperature']
    #era5_atmosphere_layers_v2['h2o'] = era5_atmosphere_layers_year['h2o']

    optical_props = gas_optics_lw.compute(
        era5_atmosphere_layers_v2, 
        add_to_input=False,
    )

    optical_props["surface_emissivity"] = 0.96

    clr_fluxes = optical_props.rte.solve(add_to_input=False)
    lwd_sfc_clr_yr_q1980.append(clr_fluxes.sel(level=0, column=0).lw_flux_down.values.item())

1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [213]:
# original temperature and q profiles preserved
lwd_sfc_clr_yr = []
for year in era5_atmosphere_layers.year:
    print(year.item())
    era5_atmosphere_layers_year = era5_atmosphere_layers.sel(year=year)
    era5_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
    era5_atmosphere_layers_v2['pres_level'] = xr.ones_like(era5_atmosphere_layers_v2['pres_level']) * era5_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
    era5_atmosphere_layers_v2['pres_layer'] = era5_atmosphere_layers_year['pres_layer']
    era5_atmosphere_layers_v2['temp_level'] = xr.ones_like(era5_atmosphere_layers_v2['temp_level']) * era5_atmosphere_layers_year['temp_level'].values
    era5_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_year['o3']
    era5_atmosphere_layers_v2['temp_layer'] = era5_atmosphere_layers_year['temp_layer']
    era5_atmosphere_layers_v2['surface_temperature'] = era5_atmosphere_layers_year['surface_temperature']
    era5_atmosphere_layers_v2['h2o'] = era5_atmosphere_layers_year['h2o']
    #era5_atmosphere_layers_v2['co2'] = 0.000373

    optical_props = gas_optics_lw.compute(
        era5_atmosphere_layers_v2, 
        add_to_input=False,
    )

    optical_props["surface_emissivity"] = 0.96

    clr_fluxes = optical_props.rte.solve(add_to_input=False)
    lwd_sfc_clr_yr.append(clr_fluxes.sel(level=0, column=0).lw_flux_down.values.item())

1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


### CMIP6 runs

In [222]:
# open model ensemble members used in analysis
with open('/home/tessj/wus_temp_trends/modmembers_allvars_WUS_subset_forpyrte.json', 'r') as file:
        modmembers_all = json.load(file)

In [224]:
# open CMIP6 temperature, humidity, longwave, skin temperature
cmip6_ta = xr.open_dataset('/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/ta_WUSavg_19800101-20241231_allmodels.nc').resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')
cmip6_hus = xr.open_dataset('/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/hus_WUSavg_19800101-20241231_allmodels.nc').resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')
#cmip6_zg = xr.open_dataset('/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/zg_WUSavg_19800101-20241231_allmodels.nc')

In [225]:
cmip6_lwd = xr.open_dataset('/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/rlds_WUSavg_19800101-20241231_allmodels.nc').resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')
cmip6_lwd_clr = xr.open_dataset('/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/rldscs_WUSavg_19800101-20241231_allmodels.nc').resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')


In [68]:
cmip6_skt = xr.open_dataset('/d5/tessj/data/CMIP6/for_tess/WUS_subset_forpyrte/ts_WUSavg_19800101-20241231_allmodels.nc').resample(time='QS-JAN').mean('time').isel(time=slice(0,None,4)).groupby('time.year').mean('time')


In [190]:
# original CMIP6 temperature and humidity data
for model in list(modmembers_all.keys())[:]:
    print(model)
    cmip6_lwd_sfc_clr_members = [] 
    for member_id in modmembers_all[model]:
        model_ta = cmip6_ta.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_hus = cmip6_hus.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_skt = cmip6_skt.sel(model=model, member_id=member_id)

        cmip6_atmosphere = model_ta

        cmip6_atmosphere['h2o'] = (model_hus.hus/((1 - model_hus.hus)*0.622))
        cmip6_atmosphere['surface_temperature'] = model_skt.ts
        cmip6_atmosphere['pres_layer'] = cmip6_atmosphere.plev

        cmip6_atmosphere = cmip6_atmosphere.assign_coords({'plev':atmosphere.layer.values[:21]})
        cmip6_atmosphere = cmip6_atmosphere.rename({'plev':'layer'})

        cmip6_atmosphere_layer = cmip6_atmosphere.isel(layer=slice(1, None, 2)).assign_coords({'layer':np.arange(0,10)}).rename({'ta':'temp_layer'})
        cmip6_atmosphere_level = cmip6_atmosphere.isel(layer=slice(0, None, 2)).assign_coords({'layer':np.arange(0,11)}).rename({'layer':'level', 'ta':'temp_level', 'pres_layer':'pres_level'})
        cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
        cmip6_atmosphere_layers = xr.merge([cmip6_atmosphere_layer, cmip6_atmosphere_level])
        cmip6_atmosphere_layers = xr.concat([cmip6_atmosphere_layers.assign_coords({'column':0}),cmip6_atmosphere_layers.assign_coords({'column':1})], dim='column')

        cmip6_lwd_sfc_clr_yr = []
        for year in era5_atmosphere_layers.year:
            #print(year.item())
            cmip6_atmosphere_layers_year = cmip6_atmosphere_layers.sel(year=year)
            cmip6_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
            cmip6_atmosphere_layers_v2['pres_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['pres_level']) * cmip6_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
            cmip6_atmosphere_layers_v2['pres_layer'] = cmip6_atmosphere_layers_year['pres_layer']
            cmip6_atmosphere_layers_v2['temp_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['temp_level']) * cmip6_atmosphere_layers_year['temp_level'].values
            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']
            cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_year['temp_layer']
            cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
            cmip6_atmosphere_layers_v2['h2o'] = cmip6_atmosphere_layers_year['h2o']
            
            # if there are nans at the lower levels, eliminate those levels/layers before calculating
            nancount = sum(np.isnan(cmip6_atmosphere_layers_v2.sel(column=0).temp_layer)).item()
            cmip6_atmosphere_layers_v2 = cmip6_atmosphere_layers_v2.sel(layer=slice(nancount, 9), level=slice(nancount,11))

            optical_props = gas_optics_lw.compute(
                cmip6_atmosphere_layers_v2, 
                add_to_input=False,
            )

            optical_props["surface_emissivity"] = 0.96

            clr_fluxes = optical_props.rte.solve(add_to_input=False)
            #break
            cmip6_lwd_sfc_clr_yr.append(clr_fluxes.sel(level=nancount, column=0).lw_flux_down.values.item())

        cmip6_lwd_sfc_clr = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
        cmip6_lwd_sfc_clr_members.append(cmip6_lwd_sfc_clr)
        
    cmip6_lwd_sfc_clr_model = xr.concat(cmip6_lwd_sfc_clr_members, dim='member_id')
    cmip6_lwd_sfc_clr_model.to_netcdf(f'./CMIP6_LWD_sfc_clr/{model}_lwd_members.nc')
    #cmip6_lwd_sfc_clr_models.append(cmip6_lwd_sfc_clr_model)

    

CNRM-CM6-1


/tmp/ipykernel_15183/1124941475.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_15183/1124941475.py:54: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr


ValueError: operands could not be broadcast together with shapes (45,) (0,) 

In [ ]:
# hold temperature at 1980 value
#cmip6_lwd_sfc_clr_models = []
for model in list(modmembers_all.keys())[:]:
    print(model)
    cmip6_lwd_sfc_clr_members = [] 
    for member_id in modmembers_all[model]:
        model_ta = cmip6_ta.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_hus = cmip6_hus.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_skt = cmip6_skt.sel(model=model, member_id=member_id)

        cmip6_atmosphere = model_ta

        cmip6_atmosphere['h2o'] = (model_hus.hus/((1 - model_hus.hus)*0.622))
        cmip6_atmosphere['surface_temperature'] = model_skt.ts

        cmip6_atmosphere['pres_layer'] = cmip6_atmosphere.plev

        cmip6_atmosphere = cmip6_atmosphere.assign_coords({'plev':atmosphere.layer.values[:21]})
        cmip6_atmosphere = cmip6_atmosphere.rename({'plev':'layer'})

        cmip6_atmosphere_layer = cmip6_atmosphere.isel(layer=slice(1, None, 2)).assign_coords({'layer':np.arange(0,10)}).rename({'ta':'temp_layer'})
        cmip6_atmosphere_level = cmip6_atmosphere.isel(layer=slice(0, None, 2)).assign_coords({'layer':np.arange(0,11)}).rename({'layer':'level', 'ta':'temp_level', 'pres_layer':'pres_level'})
        cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
        cmip6_atmosphere_layers = xr.merge([cmip6_atmosphere_layer, cmip6_atmosphere_level])
        cmip6_atmosphere_layers = xr.concat([cmip6_atmosphere_layers.assign_coords({'column':0}),cmip6_atmosphere_layers.assign_coords({'column':1})], dim='column')

        cmip6_lwd_sfc_clr_yr = []
        for year in era5_atmosphere_layers.year:
            #print(year.item())
            cmip6_atmosphere_layers_year = cmip6_atmosphere_layers.sel(year=year)
            cmip6_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
            cmip6_atmosphere_layers_v2['pres_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['pres_level']) * cmip6_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
            cmip6_atmosphere_layers_v2['pres_layer'] = cmip6_atmosphere_layers_year['pres_layer']
            if year == 1980:
                cmip6_atmosphere_layers_v2['temp_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['temp_level']) * cmip6_atmosphere_layers_year['temp_level'].values
                cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_year['temp_layer']
                cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
                cmip6_atmosphere_layers_v2_templevel_1980 = cmip6_atmosphere_layers_v2['temp_level']
                cmip6_atmosphere_layers_v2_templayer_1980 = cmip6_atmosphere_layers_v2['temp_layer']
                cmip6_atmosphere_layers_v2_surftemp_1980 = cmip6_atmosphere_layers_v2['surface_temperature']

            else:
                cmip6_atmosphere_layers_v2['temp_level'] = cmip6_atmosphere_layers_v2_templevel_1980
                cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_v2_templayer_1980                
                cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_v2_surftemp_1980                    

            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']
            #cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
            cmip6_atmosphere_layers_v2['h2o'] = cmip6_atmosphere_layers_year['h2o']
            
            # if there are nans at the lower levels, eliminate those levels/layers before calculating
            nancount = sum(np.isnan(cmip6_atmosphere_layers_v2.sel(column=0).temp_layer)).item()
            cmip6_atmosphere_layers_v2 = cmip6_atmosphere_layers_v2.sel(layer=slice(nancount, 9), level=slice(nancount,11))

            optical_props = gas_optics_lw.compute(
                cmip6_atmosphere_layers_v2, 
                add_to_input=False,
            )

            optical_props["surface_emissivity"] = 0.96

            clr_fluxes = optical_props.rte.solve(add_to_input=False)
            #break
            cmip6_lwd_sfc_clr_yr.append(clr_fluxes.sel(level=nancount, column=0).lw_flux_down.values.item())

        cmip6_lwd_sfc_clr_tempconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
        cmip6_lwd_sfc_clr_members.append(cmip6_lwd_sfc_clr_tempconst)
        
    cmip6_lwd_sfc_clr_model = xr.concat(cmip6_lwd_sfc_clr_members, dim='member_id')
    cmip6_lwd_sfc_clr_model.to_netcdf(f'./CMIP6_LWD_sfc_clr/{model}_lwd_members_temp1980.nc')
    #cmip6_lwd_sfc_clr_models.append(cmip6_lwd_sfc_clr_model)

 

In [97]:
# detrend temperatures
for model in list(modmembers_all.keys())[0:]:
    print(model)
    cmip6_lwd_sfc_clr_members = [] 
    for member_id in modmembers_all[model]:
        model_ta = cmip6_ta.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_hus = cmip6_hus.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_skt = cmip6_skt.sel(model=model, member_id=member_id)

        cmip6_atmosphere = model_ta

        cmip6_atmosphere['h2o'] = (model_hus.hus/((1 - model_hus.hus)*0.622))
        cmip6_atmosphere['surface_temperature'] = model_skt.ts

        cmip6_atmosphere['pres_layer'] = cmip6_atmosphere.plev

        cmip6_atmosphere = cmip6_atmosphere.assign_coords({'plev':atmosphere.layer.values[:21]})
        cmip6_atmosphere = cmip6_atmosphere.rename({'plev':'layer'})

        cmip6_atmosphere_layer = cmip6_atmosphere.isel(layer=slice(1, None, 2)).assign_coords({'layer':np.arange(0,10)}).rename({'ta':'temp_layer'})
        cmip6_atmosphere_level = cmip6_atmosphere.isel(layer=slice(0, None, 2)).assign_coords({'layer':np.arange(0,11)}).rename({'layer':'level', 'ta':'temp_level', 'pres_layer':'pres_level'})
        cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
        cmip6_atmosphere_layers = xr.merge([cmip6_atmosphere_layer, cmip6_atmosphere_level])
        cmip6_atmosphere_layers = xr.concat([cmip6_atmosphere_layers.assign_coords({'column':0}),cmip6_atmosphere_layers.assign_coords({'column':1})], dim='column')
        
        cmip6_templevel_mean = cmip6_atmosphere_layers['temp_level'].mean('year')
        cmip6_templevel_detrend = detrend_dim(cmip6_atmosphere_layers['temp_level'], dim='year') + cmip6_templevel_mean
        cmip6_atmosphere_layers['temp_level'] = cmip6_templevel_detrend
        
        cmip6_templayer_mean = cmip6_atmosphere_layers['temp_layer'].mean('year')
        cmip6_templayer_detrend = detrend_dim(cmip6_atmosphere_layers['temp_layer'], dim='year') + cmip6_templayer_mean
        cmip6_atmosphere_layers['temp_layer'] = cmip6_templayer_detrend
        
        cmip6_surftemp_mean = cmip6_atmosphere_layers['surface_temperature'].mean('year')
        cmip6_surftemp_detrend = detrend_dim(cmip6_atmosphere_layers['surface_temperature'], dim='year') + cmip6_surftemp_mean
        cmip6_atmosphere_layers['surface_temperature'] = cmip6_surftemp_detrend
        
        cmip6_lwd_sfc_clr_yr = []
        for year in era5_atmosphere_layers.year:
            #print(year.item())
            cmip6_atmosphere_layers_year = cmip6_atmosphere_layers.sel(year=year)
            cmip6_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
            cmip6_atmosphere_layers_v2['pres_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['pres_level']) * cmip6_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
            cmip6_atmosphere_layers_v2['pres_layer'] = cmip6_atmosphere_layers_year['pres_layer']
            cmip6_atmosphere_layers_v2['temp_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['temp_level']) * cmip6_atmosphere_layers_year['temp_level'].values
            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']
            cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_year['temp_layer']
            cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
            cmip6_atmosphere_layers_v2['h2o'] = cmip6_atmosphere_layers_year['h2o']                  
            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']
            
            # if there are nans at the lower levels, eliminate those levels/layers before calculating
            nancount = sum(np.isnan(cmip6_atmosphere_layers_v2.sel(column=0).temp_layer)).item()
            cmip6_atmosphere_layers_v2 = cmip6_atmosphere_layers_v2.sel(layer=slice(nancount, 9), level=slice(nancount,11))

            optical_props = gas_optics_lw.compute(
                cmip6_atmosphere_layers_v2, 
                add_to_input=False,
            )

            optical_props["surface_emissivity"] = 0.96

            clr_fluxes = optical_props.rte.solve(add_to_input=False)
            #break
            cmip6_lwd_sfc_clr_yr.append(clr_fluxes.sel(level=nancount, column=0).lw_flux_down.values.item())

        cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
        cmip6_lwd_sfc_clr_members.append(cmip6_lwd_sfc_clr_tempdetrend)
        
    cmip6_lwd_sfc_clr_model = xr.concat(cmip6_lwd_sfc_clr_members, dim='member_id')
    cmip6_lwd_sfc_clr_model.to_netcdf(f'./CMIP6_LWD_sfc_clr/{model}_lwd_members_tempdetrend.nc')
    #cmip6_lwd_sfc_clr_models.append(cmip6_lwd_sfc_clr_model)

 

GFDL-ESM4


/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping v

IPSL-CM6A-LR


/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping v

CNRM-CM6-1


...

/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping v

CESM2


/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping v

EC-Earth3-Veg-LR


/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3651323325.py:22: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3651323325.py:67: FutureWarning: dropping v

In [109]:
# detrend temp and prescribe era5 q trend
#cmip6_lwd_sfc_clr_models = []
for model in list(modmembers_all.keys())[:]:
    print(model)
    cmip6_lwd_sfc_clr_members = [] 
    for member_id in modmembers_all[model]:
        model_ta = cmip6_ta.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_hus = cmip6_hus.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_skt = cmip6_skt.sel(model=model, member_id=member_id)

        cmip6_atmosphere = model_ta

        cmip6_atmosphere['h2o'] = (model_hus.hus/((1 - model_hus.hus)*0.622))
        cmip6_atmosphere['surface_temperature'] = model_skt.ts

        cmip6_atmosphere['pres_layer'] = cmip6_atmosphere.plev

        cmip6_atmosphere = cmip6_atmosphere.assign_coords({'plev':atmosphere.layer.values[:21]})
        cmip6_atmosphere = cmip6_atmosphere.rename({'plev':'layer'})

        cmip6_atmosphere_layer = cmip6_atmosphere.isel(layer=slice(1, None, 2)).assign_coords({'layer':np.arange(0,10)}).rename({'ta':'temp_layer'})
        cmip6_atmosphere_level = cmip6_atmosphere.isel(layer=slice(0, None, 2)).assign_coords({'layer':np.arange(0,11)}).rename({'layer':'level', 'ta':'temp_level', 'pres_layer':'pres_level'})
        cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
        cmip6_atmosphere_layers = xr.merge([cmip6_atmosphere_layer, cmip6_atmosphere_level])
        cmip6_atmosphere_layers = xr.concat([cmip6_atmosphere_layers.assign_coords({'column':0}),cmip6_atmosphere_layers.assign_coords({'column':1})], dim='column')
        
        cmip6_templevel_mean = cmip6_atmosphere_layers['temp_level'].mean('year')
        cmip6_templevel_detrend = detrend_dim(cmip6_atmosphere_layers['temp_level'], dim='year') + cmip6_templevel_mean
        cmip6_atmosphere_layers['temp_level'] = cmip6_templevel_detrend
        
        cmip6_templayer_mean = cmip6_atmosphere_layers['temp_layer'].mean('year')
        cmip6_templayer_detrend = detrend_dim(cmip6_atmosphere_layers['temp_layer'], dim='year') + cmip6_templayer_mean
        cmip6_atmosphere_layers['temp_layer'] = cmip6_templayer_detrend
        
        cmip6_surftemp_mean = cmip6_atmosphere_layers['surface_temperature'].mean('year')
        cmip6_surftemp_detrend = detrend_dim(cmip6_atmosphere_layers['surface_temperature'], dim='year') + cmip6_surftemp_mean
        cmip6_atmosphere_layers['surface_temperature'] = cmip6_surftemp_detrend
        
        cmip6_h2o_mean = cmip6_atmosphere_layers['h2o'].mean('year')
        cmip6_h2o_with_era5_trend = detrend_dim(cmip6_atmosphere_layers['h2o'], dim='year') + era5_h2o_trend - era5_h2o_trend.mean('year') + cmip6_h2o_mean
        cmip6_atmosphere_layers['h2o'] = cmip6_h2o_with_era5_trend
        
        cmip6_lwd_sfc_clr_yr = []
        for year in era5_atmosphere_layers.year:
            #print(year.item())
            cmip6_atmosphere_layers_year = cmip6_atmosphere_layers.sel(year=year)
            cmip6_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
            cmip6_atmosphere_layers_v2['pres_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['pres_level']) * cmip6_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
            cmip6_atmosphere_layers_v2['pres_layer'] = cmip6_atmosphere_layers_year['pres_layer']
            cmip6_atmosphere_layers_v2['temp_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['temp_level']) * cmip6_atmosphere_layers_year['temp_level'].values
            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']
            cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_year['temp_layer']
            cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
            cmip6_atmosphere_layers_v2['h2o'] = cmip6_atmosphere_layers_year['h2o']                  
            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']#cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']

            # if there are nans at the lower levels, eliminate those levels/layers before calculating
            nancount = sum(np.isnan(cmip6_atmosphere_layers_v2.sel(column=0).temp_layer)).item()
            cmip6_atmosphere_layers_v2 = cmip6_atmosphere_layers_v2.sel(layer=slice(nancount, 9), level=slice(nancount,11))

            optical_props = gas_optics_lw.compute(
                cmip6_atmosphere_layers_v2, 
                add_to_input=False,
            )

            optical_props["surface_emissivity"] = 0.96

            clr_fluxes = optical_props.rte.solve(add_to_input=False)
            #break
            cmip6_lwd_sfc_clr_yr.append(clr_fluxes.sel(level=nancount, column=0).lw_flux_down.values.item())

        cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
        cmip6_lwd_sfc_clr_members.append(cmip6_lwd_sfc_clr_tempdetrend_era5h2o)
        
    cmip6_lwd_sfc_clr_model = xr.concat(cmip6_lwd_sfc_clr_members, dim='member_id')
    cmip6_lwd_sfc_clr_model.to_netcdf(f'./CMIP6_LWD_sfc_clr/{model}_lwd_members_tempdetrend_era5h2otrend.nc')
    #cmip6_lwd_sfc_clr_models.append(cmip6_lwd_sfc_clr_model)

 

GFDL-ESM4


/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWar

IPSL-CM6A-LR


/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWar

CNRM-CM6-1


...

/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWar

CESM2


/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWar

EC-Earth3-Veg-LR


/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempdetrend_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/3526391619.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/3526391619.py:72: FutureWar

In [82]:
# hold temp constant at 1980 value and prescribe era5 q trend
#cmip6_lwd_sfc_clr_models = []
for model in list(modmembers_all.keys())[:]:
    print(model)
    cmip6_lwd_sfc_clr_members = [] 
    for member_id in modmembers_all[model]:
        model_ta = cmip6_ta.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_hus = cmip6_hus.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_skt = cmip6_skt.sel(model=model, member_id=member_id)

        cmip6_atmosphere = model_ta

        cmip6_atmosphere['h2o'] = (model_hus.hus/((1 - model_hus.hus)*0.622))
        cmip6_atmosphere['surface_temperature'] = model_skt.ts

        cmip6_atmosphere['pres_layer'] = cmip6_atmosphere.plev

        cmip6_atmosphere = cmip6_atmosphere.assign_coords({'plev':atmosphere.layer.values[:21]})
        cmip6_atmosphere = cmip6_atmosphere.rename({'plev':'layer'})

        cmip6_atmosphere_layer = cmip6_atmosphere.isel(layer=slice(1, None, 2)).assign_coords({'layer':np.arange(0,10)}).rename({'ta':'temp_layer'})
        cmip6_atmosphere_level = cmip6_atmosphere.isel(layer=slice(0, None, 2)).assign_coords({'layer':np.arange(0,11)}).rename({'layer':'level', 'ta':'temp_level', 'pres_layer':'pres_level'})
        cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
        cmip6_atmosphere_layers = xr.merge([cmip6_atmosphere_layer, cmip6_atmosphere_level])
        cmip6_atmosphere_layers = xr.concat([cmip6_atmosphere_layers.assign_coords({'column':0}),cmip6_atmosphere_layers.assign_coords({'column':1})], dim='column')
        cmip6_h2o_mean = cmip6_atmosphere_layers['h2o'].mean('year')
        cmip6_h2o_with_era5_trend = detrend_dim(cmip6_atmosphere_layers['h2o'], dim='year') + era5_h2o_trend - era5_h2o_trend.mean('year') + cmip6_h2o_mean
        cmip6_atmosphere_layers['h2o'] = cmip6_h2o_with_era5_trend
        cmip6_lwd_sfc_clr_yr = []
        for year in era5_atmosphere_layers.year:
            #print(year.item())
            cmip6_atmosphere_layers_year = cmip6_atmosphere_layers.sel(year=year)
            cmip6_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
            cmip6_atmosphere_layers_v2['pres_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['pres_level']) * cmip6_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
            cmip6_atmosphere_layers_v2['pres_layer'] = cmip6_atmosphere_layers_year['pres_layer']
            if year == 1980:
                cmip6_atmosphere_layers_v2['temp_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['temp_level']) * cmip6_atmosphere_layers_year['temp_level'].values
                cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_year['temp_layer']
                cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
                cmip6_atmosphere_layers_v2_templevel_1980 = cmip6_atmosphere_layers_v2['temp_level']
                cmip6_atmosphere_layers_v2_templayer_1980 = cmip6_atmosphere_layers_v2['temp_layer']
                cmip6_atmosphere_layers_v2_surftemp_1980 = cmip6_atmosphere_layers_v2['surface_temperature']

            else:
                cmip6_atmosphere_layers_v2['temp_level'] = cmip6_atmosphere_layers_v2_templevel_1980
                cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_v2_templayer_1980                
                cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_v2_surftemp_1980                

            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']
            #cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
            cmip6_atmosphere_layers_v2['h2o'] = cmip6_atmosphere_layers_year['h2o']
            # if there are nans at the lower levels, eliminate those levels/layers before calculating
            nancount = sum(np.isnan(cmip6_atmosphere_layers_v2.sel(column=0).temp_layer)).item()
            cmip6_atmosphere_layers_v2 = cmip6_atmosphere_layers_v2.sel(layer=slice(nancount, 9), level=slice(nancount,11))

            optical_props = gas_optics_lw.compute(
                cmip6_atmosphere_layers_v2, 
                add_to_input=False,
            )

            optical_props["surface_emissivity"] = 0.96

            clr_fluxes = optical_props.rte.solve(add_to_input=False)
            #break
            cmip6_lwd_sfc_clr_yr.append(clr_fluxes.sel(level=nancount, column=0).lw_flux_down.values.item())

        cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
        cmip6_lwd_sfc_clr_members.append(cmip6_lwd_sfc_clr_tempconst_era5h2o)
        
    cmip6_lwd_sfc_clr_model = xr.concat(cmip6_lwd_sfc_clr_members, dim='member_id')
    cmip6_lwd_sfc_clr_model.to_netcdf(f'./CMIP6_LWD_sfc_clr/{model}_lwd_members_temp1980_era5h2otrend.nc')
    #cmip6_lwd_sfc_clr_models.append(cmip6_lwd_sfc_clr_model)

 

GFDL-ESM4


/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning

IPSL-CM6A-LR


/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning

CNRM-CM6-1


...

/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning

CESM2


/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning

EC-Earth3-Veg-LR


/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_tempconst_era5h2o = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_27844/1907837334.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_27844/1907837334.py:67: FutureWarning

In [51]:
# hold q at 1980 value
#cmip6_lwd_sfc_clr_models = []
for model in list(modmembers_all.keys())[:]:
    print(model)
    cmip6_lwd_sfc_clr_members = [] 
    for member_id in modmembers_all[model]:
        model_ta = cmip6_ta.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_hus = cmip6_hus.interp(plev = (100*era5_jfm.pressure_level).values).sel(model=model, member_id=member_id)
        model_skt = cmip6_skt.sel(model=model, member_id=member_id)

        cmip6_atmosphere = model_ta

        cmip6_atmosphere['h2o'] = (model_hus.hus/((1 - model_hus.hus)*0.622))
        cmip6_atmosphere['surface_temperature'] = model_skt.ts

        cmip6_atmosphere['pres_layer'] = cmip6_atmosphere.plev

        cmip6_atmosphere = cmip6_atmosphere.assign_coords({'plev':atmosphere.layer.values[:21]})
        cmip6_atmosphere = cmip6_atmosphere.rename({'plev':'layer'})

        cmip6_atmosphere_layer = cmip6_atmosphere.isel(layer=slice(1, None, 2)).assign_coords({'layer':np.arange(0,10)}).rename({'ta':'temp_layer'})
        cmip6_atmosphere_level = cmip6_atmosphere.isel(layer=slice(0, None, 2)).assign_coords({'layer':np.arange(0,11)}).rename({'layer':'level', 'ta':'temp_level', 'pres_layer':'pres_level'})
        cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
        cmip6_atmosphere_layers = xr.merge([cmip6_atmosphere_layer, cmip6_atmosphere_level])
        cmip6_atmosphere_layers = xr.concat([cmip6_atmosphere_layers.assign_coords({'column':0}),cmip6_atmosphere_layers.assign_coords({'column':1})], dim='column')

        cmip6_lwd_sfc_clr_yr = []
        for year in era5_atmosphere_layers.year:
            #print(year.item())
            cmip6_atmosphere_layers_year = cmip6_atmosphere_layers.sel(year=year)
            cmip6_atmosphere_layers_v2 = ex_atmosphere.sel(column=slice(0,2), layer = slice(0,10), level=slice(0,11))
            cmip6_atmosphere_layers_v2['pres_level'] = xr.ones_like(cmip6_atmosphere_layers_v2['pres_level']) * cmip6_atmosphere_layers_year['pres_level'].values#era5_atmosphere_layers_v2['pres_level'] - 100
            cmip6_atmosphere_layers_v2['pres_layer'] = cmip6_atmosphere_layers_year['pres_layer']
            if year == 1980:
                cmip6_atmosphere_layers_v2['h2o'] = xr.ones_like(cmip6_atmosphere_layers_v2['h2o']) * cmip6_atmosphere_layers_year['h2o'].values
                #cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_year['temp_layer']
                #cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
                cmip6_atmosphere_layers_v2_h2olayer_1980 = cmip6_atmosphere_layers_v2['h2o']
                #cmip6_atmosphere_layers_v2_templayer_1980 = cmip6_atmosphere_layers_v2['temp_layer']
                #cmip6_atmosphere_layers_v2_surftemp_1980 = cmip6_atmosphere_layers_v2['surface_temperature']

            else:
                cmip6_atmosphere_layers_v2['h2o'] = cmip6_atmosphere_layers_v2_h2olayer_1980

                #cmip6_atmosphere_layers_v2['temp_level'] = cmip6_atmosphere_layers_v2_templevel_1980
                #cmip6_atmosphere_layers_v2['temp_layer'] = cmip6_atmosphere_layers_v2_templayer_1980                
                #cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_v2_surftemp_1980                    
            
            cmip6_atmosphere_layers_v2['temp_layer'] = era5_atmosphere_layers_v2['temp_layer']
            cmip6_atmosphere_layers_v2['surface_temperature'] = era5_atmosphere_layers_v2['surface_temperature']
            cmip6_atmosphere_layers_v2['o3'] = era5_atmosphere_layers_v2['o3']
            cmip6_atmosphere_layers_v2['surface_temperature'] = cmip6_atmosphere_layers_year['surface_temperature']
            cmip6_atmosphere_layers_v2['h2o'] = cmip6_atmosphere_layers_year['h2o']
            
            # if there are nans at the lower levels, eliminate those levels/layers before calculating
            nancount = sum(np.isnan(cmip6_atmosphere_layers_v2.sel(column=0).temp_layer)).item()
            cmip6_atmosphere_layers_v2 = cmip6_atmosphere_layers_v2.sel(layer=slice(nancount, 9), level=slice(nancount,11))

            optical_props = gas_optics_lw.compute(
                cmip6_atmosphere_layers_v2, 
                add_to_input=False,
            )

            optical_props["surface_emissivity"] = 0.96

            clr_fluxes = optical_props.rte.solve(add_to_input=False)
            #break
            cmip6_lwd_sfc_clr_yr.append(clr_fluxes.sel(level=nancount, column=0).lw_flux_down.values.item())

        cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
        cmip6_lwd_sfc_clr_members.append(cmip6_lwd_sfc_clr_qconst)
        
    cmip6_lwd_sfc_clr_model = xr.concat(cmip6_lwd_sfc_clr_members, dim='member_id')
    cmip6_lwd_sfc_clr_model.to_netcdf(f'./CMIP6_LWD_sfc_clr/{model}_lwd_members_q1980.nc')
    #cmip6_lwd_sfc_clr_models.append(cmip6_lwd_sfc_clr_model)

 

/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])


GFDL-ESM4


/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables u

IPSL-CM6A-LR


/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables u

...

/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables u

CESM2


/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables u

EC-Earth3-Veg-LR


/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_lwd_sfc_clr_qconst = xr.zeros_like(cmip6_atmosphere_layers['h2o'].sel(layer=0, column=0).drop(['column','layer'])) +cmip6_lwd_sfc_clr_yr
/tmp/ipykernel_22931/3905396680.py:23: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  cmip6_atmosphere_level = cmip6_atmosphere_level.drop(['surface_temperature','h2o'])
/tmp/ipykernel_22931/3905396680.py:70: FutureWarning: dropping variables u

In [108]:
# pyRTE clr LWD trend given detrended ERA5 temp and ERA5 q
45*np.polyfit(era5_atmosphere_layers.year,lwd_sfc_clr_yr_tempdetrend, deg=1)[0]

np.float64(-2.263649909123405)

In [61]:
# pyRTE clr LWD trend given constant 1980 ERA5 temp and ERA5 q

45*np.polyfit(era5_atmosphere_layers.year, lwd_sfc_clr_yr_temp1980, deg=1)[0]

np.float64(-2.2495023118862996)

In [41]:
# pyRTE clr LWD trend given constant 1980 ERA5 q and ERA5 T

45*np.polyfit(era5_atmosphere_layers.year,lwd_sfc_clr_yr_q1980, deg=1)[0]

np.float64(0.3203681778031547)